# Exclusive-fate spline models across neighborhood radii

**Role:** Training.

Fit interpretable, additive curvature models with smooth coefficients in log cell count N. Compare center identity alone (radius 0), shared neighborhood composition, and center-dependent neighborhood composition at radii 1–4. Every shell contains unique cells at exactly that shortest-path distance; shells 1 through the configured radius all enter the prediction. Fate exclusivity is mandatory, and Unassigned is a separate identity.

Coefficients add directly in physical curvature units. Optional residualization uses a saved organoid baseline; there is no asinh output transformation or MLP. Neighborhood fractions are centered on a training-derived reference. This first version assumes linear responses to composition; it does not yet fit nonlinear abundance/saturation or a separate ring-size effect. Empty shells contribute zero.

A completed reference run supplies the exact cohort, outer folds, cleaned targets and fitted baselines. This notebook cannot create or refilter a cohort, resplit the outer folds, or fit a new baseline. Inner organoid holdouts select spline penalties; the outer validation folds are used only for reporting. Artifacts include models, physical targets, split membership, baselines, reference compositions, knots, penalty trials and settings. Nothing trains until `RUN_TRAINING=True`. Separate notebooks compare MSE and inspect coefficients without refitting.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/models/gnn.py').is_file())  # Repository root.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Resolve data and saved-run paths relative to this directory.

## Configuration

Choose the required source run, model radii, baseline subtraction and spline penalties. Dataset, target selection, timepoints, filters, outer splits and baseline training settings are inherited exclusively from the reference run. Its fitted baselines are copied without retraining. It does not copy GIN architecture or loss settings. The default Ndependent run supplies exclusive fates, all saved timepoints, the exact outer train/validation folds and saved baseline preprocessing. Loading these datasets does not select or load a reference predictor: all its masking rates share the same observed fold inputs. Choose the zero-masking FiLM model later in the comparison notebook. The observed missingness channel is removed from spline inputs, retaining Unassigned as an explicit identity. Settings actually used are written to the new run.

In [ ]:
import copy
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.artifacts.bundle import save_bundle, graph_membership
from src.artifacts.paths import training_run_path
from src.artifacts.provenance import workflow_fingerprint
from src.data.marker_exclusivity import exclusive_markers, EXCLUSIVITY_RULES
from src.data.target_transforms import IdentityTransform
from src.training.spline_fit import graph_samples, fit_spline, sample_mse
from src.training.progress import TrainingProgress
from src.analysis.metrics.evaluation import organoid_mse_summary, prediction_table, regional_mse
from src.analysis.spatial.regions import graph_regions
from src.plotting.depth_scan import plot_regional_mse

# Execution and progress
RUN_TRAINING = True  # Enable only after reviewing the cohort and settings.
SHOW_PROGRESS = True  # Persistent fold/model/penalty progress panel.

# Required source run: all data, outer splits and baselines are inherited
REFERENCE_RUN = 'training_results/fate_masking_training/Ndependent_20260922_115100'  # Reuse exact saved cohort, outer folds and baseline.

SETTINGS = dict(
    # Run identity and model grid
    tag='exclusive_pairwise',  # Optional purpose label in the timestamped directory name.
    variants=['center', 'shared', 'pairwise'],  # Nested additive models to fit.
    radii=[1, 2, 3, 4],  # Positive exact-hop radii; center-only is fitted once at radius 0.
    exclusive_markers=True,  # Required by this model; rules are saved with the run.
    seeds=[42],  # Seeds for inner organoid holdouts; the quadratic fit itself is deterministic.

    # Size smoothing and regularization
    n_splines=5,  # Cubic B-spline basis size, at least 4; uniformly spaced knots in log N.
    smoothness_grid=[.01, .1, 1.],  # Second-difference penalties tested on the inner holdout.
    ridge=.001,  # Shrinkage on center/shared coefficients (not b).
    pair_ridge_grid=[.001, .01],  # Shrinkage on center-specific neighborhood deviations.
    inner_fraction=.2,  # Fraction of outer-training organoids used for penalty selection.
    blas_threads=2,  # CPU BLAS threads for sufficient statistics and linear solves.

    # Spline target parameterization (saved physical targets and baseline stay fixed)
    residualize=True,  # Subtract the baseline; False fits physical curvature directly.
    target_scaling='identity',  # Keep terms additive in physical units; no nonlinear target transform.
)

# Regional evaluation (source annotations only; never model inputs)
EVALUATE_REGIONS = True  # Also save crypt/qualified-neck/villus MSE by radius.
REGION_SETTINGS = dict(
    crypt_max=.8,  # Upper normalized-distance boundary of the crypt body.
    neck_max=1.2,  # Upper boundary of the circumference-qualified neck band.
    neck_config=dict(
        minimum_crypt_cells=10,  # Minimum support for classifying one crypt profile.
        minimum_relative_depth=.05,  # Required relative circumference trough depth.
        plateau_max_range=.10,  # Maximum relative range of a flat neck section.
        plateau_max_slope=.25,  # Maximum slope of a flat neck section.
        plateau_exit_rise=.10,  # Required rise beyond a flat neck section.
    ),
)
RUN_DIR = training_run_path(ROOT, 'fate_spline_training', SETTINGS['tag'])  # New immutable run destination.

## Restore the reference cohort and exact folds

Reference mode loads the run's shared intact, full-marker input bundle for each fold and verifies membership against its split manifest. It does not load a reference predictor or select an architecture, seed or masking rate. If a fold has genuinely different input bundles, loading fails rather than choosing one arbitrarily. Its baseline is copied, not fitted again. Later, targets are reconstructed in physical units before the requested residualization is applied. No data filters, target cleanup or outer splitting are rerun. Dataset/filter/split metadata and baseline training settings are copied from the reference settings for provenance. Spline-specific inner holdouts only select penalties within the saved outer-training partition. No outer validation data are used to fit spline references, knots or penalties.

In [ ]:
if not SETTINGS['exclusive_markers'] or SETTINGS['target_scaling'] != 'identity':
    raise ValueError('This model requires exclusive fates and identity target scaling.')
if not SETTINGS['variants'] or len(set(SETTINGS['variants'])) != len(SETTINGS['variants']) or not set(SETTINGS['variants']) <= {'center', 'shared', 'pairwise'}:
    raise ValueError('Choose distinct center/shared/pairwise variants.')
if not SETTINGS['radii'] or any(not isinstance(r, int) or r < 1 for r in SETTINGS['radii']) or len(set(SETTINGS['radii'])) != len(SETTINGS['radii']):
    raise ValueError('radii must contain distinct positive integers.')
if not SETTINGS['seeds'] or len(set(SETTINGS['seeds'])) != len(SETTINGS['seeds']):
    raise ValueError('Choose distinct inner-split seeds.')
if not REFERENCE_RUN:
    raise ValueError('REFERENCE_RUN is required; datasets, outer folds and baselines must come from a saved run.')
reference_inputs = {}
reference = AnalysisRun(ROOT / REFERENCE_RUN)
if not reference.modern:
    raise ValueError('Reference reuse requires a modern training_results run with model/input bundles.')
membership = json.loads((reference.directory / 'splits.json').read_text())
first = reference.fold_inputs(membership[0]['fold'])
for split in membership:
    fold_data = first if split['fold'] == membership[0]['fold'] else reference.fold_inputs(split['fold'])
    if fold_data['marker_names'] != first['marker_names']:
        raise ValueError('Saved folds disagree on marker names/order.')
    reference_inputs[split['fold']] = fold_data['source_input_bundle']
marker_names = list(first['marker_names'])
graphs = list(first['raw_graphs'].values())
outlier_info = first.get('outlier_info')
effective = reference.settings
if not {'dataset', 'target_indices'} <= effective.keys():
    raise ValueError('Reference settings must record dataset and target_indices.')
for key in ('dataset', 'target_indices', 'timepoints', 'blacklist', 'sphericity_max', 'complexity_min',
            'interpolate_outliers', 'outlier_quantiles', 'n_folds', 'val_fraction', 'split_seed',
            'baseline_features', 'baseline_hidden_dim', 'baseline_max_epochs', 'baseline_patience',
            'batch_size', 'num_workers'):
    if key in effective:
        SETTINGS[key] = effective[key]
SETTINGS['reference_run'] = str(reference.directory)
SETTINGS['reference_inputs'] = reference_inputs
SETTINGS['n_folds'] = len(membership)
SETTINGS['baseline_source'] = 'copied from reference; never retrained here'
print('Inherited cohort and folds from:', reference.directory)
print('Inherited exclusive fates:', effective.get('exclusive_markers'), '| saved folds:', len(membership))
print('Spline inputs: exclusive identities, shell fractions and N; physical target residualization:', SETTINGS['residualize'])
if not marker_names or SETTINGS['target_indices'] != [0]:
    raise ValueError('Named fates and scalar Gaussian curvature are required.')
for graph in graphs:
    graph.x = torch.as_tensor(exclusive_markers(graph.x.cpu().numpy(), marker_names), dtype=graph.x.dtype)
raw_graphs = {str(g.organoid_str):g for g in graphs}
if len(raw_graphs) != len(graphs):
    raise ValueError('Duplicate organoid IDs.')
for split in membership:
    if not split['train'] or not split['validation'] or set(split['train']) & set(split['validation']):
        raise ValueError('Invalid saved train/validation membership.')
    if not (set(split['train']) | set(split['validation'])) <= raw_graphs.keys():
        raise ValueError('Split references missing organoids.')
cohort = pd.DataFrame([dict(organoid_str=g.organoid_str, n_cells=len(g.x), timepoint=str(g.meta.get('timepoint', 'unknown'))) for g in graphs])
display(cohort.groupby('timepoint').agg(organoids=('organoid_str','size'), median_N=('n_cells','median')))
display(pd.DataFrame([dict(fold=s['fold'], training=len(s['train']), validation=len(s['validation'])) for s in membership]))
model_grid = [(variant, radius) for variant in SETTINGS['variants'] for radius in ([0] if variant == 'center' else SETTINGS['radii'])]
print(f'{len(model_grid)} architectures × {len(membership)} folds × {len(SETTINGS["seeds"])} inner seeds; output: {RUN_DIR}')

## Fit and save the model grid

Each fold stores the baseline and an identity target transform. The reference run's original target transform is inverted and its baseline offset restored before optional subtraction of its saved baseline prediction. No baseline is added twice, and its constructor/weights are copied into this run's own input bundle.

Training minimizes the mean of per-organoid MSEs, so large organoids do not dominate. Smoothness penalizes second differences of spline coefficients; ridge penalties stabilize correlated fates and shrink pair terms toward the shared response. Penalties are selected on an inner organoid holdout, then the model is refitted on all outer-training organoids. Knot bounds and reference weights are refitted using only those training inputs. N outside the fitted range is clamped; there is no uncontrolled spline extrapolation.

The center-only model is fitted once per fold/seed. Increasing radius adds a separate term for each shell; it is not another GNN layer. Tables retain each fold, seed, variant and radius. The baseline is saved regardless of residualization. No production training has been executed when this notebook is created.

In [ ]:
if not RUN_TRAINING:
    raise RuntimeError('Review settings and cohort, then set RUN_TRAINING=True to fit a new run.')
RUN_DIR.mkdir(parents=True, exist_ok=False)
SETTINGS['global_features'] = []  # No head vector; the model reads size from full_num_cells.
SETTINGS['size_input'] = 'full_num_cells'  # Saved physical count, not a standardized global column.
SETTINGS['depths'] = sorted({r for _, r in model_grid})
(RUN_DIR / 'settings.json').write_text(json.dumps(SETTINGS, indent=2))
(RUN_DIR / 'splits.json').write_text(json.dumps(membership, indent=2))
(RUN_DIR / 'exclusivity_rules.json').write_text(json.dumps(EXCLUSIVITY_RULES, indent=2))
cohort.to_csv(RUN_DIR / 'cohort.csv', index=False)
provenance = {'code_sha256':workflow_fingerprint(ROOT / 'experiments/training/fate_spline_training.ipynb')}
(RUN_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2))
(RUN_DIR / 'reference_settings.json').write_text(json.dumps(reference.settings, indent=2))
source_paths = ['settings.json', 'splits.json', 'cohort/manifest.json']
source_paths.extend(f'{path}/manifest.json' for path in reference_inputs.values())
source_hashes = {path:hashlib.sha256((reference.directory/path).read_bytes()).hexdigest() for path in sorted(set(source_paths))}
(RUN_DIR / 'reference_artifacts.json').write_text(json.dumps(dict(run=str(reference.directory), sha256=source_hashes), indent=2))
save_bundle(RUN_DIR / 'cohort', dict(raw_graphs=raw_graphs, all_marker_names=marker_names,
    settings=SETTINGS, outlier_info=outlier_info), splits=membership, provenance=provenance)
records, scores, baseline_rows, tuning_parts = [], [], [], []
# Cache only immutable shell counts across folds; targets remain fold-specific.
count_cache = {}
with TrainingProgress(len(model_grid)*len(membership)*len(SETTINGS['seeds']),
                      total_baselines=len(membership), show=SHOW_PROGRESS,
                      log_path=RUN_DIR / 'training_progress.csv') as progress:
    for split in membership:
        fold = split['fold']
        with progress.task(kind='baseline', model='Copy saved baseline', fold=fold):
            source = reference.fold_inputs(fold)
            if source.get('baseline') is None or source.get('baseline_predictions') is None:
                raise ValueError('Reference must contain a fitted baseline and saved per-node baseline predictions.')
            if graph_membership(train=source['groups']['train'], validation=source['groups']['val']) != {k:split[k] for k in ('train','validation')}:
                raise ValueError('Fold inputs differ from the reference split manifest.')
            groups, offsets, baseline_validation = {}, {}, []
            for role, source_graphs in source['groups'].items():
                groups[role] = []
                for original in source_graphs:
                    g = copy.deepcopy(original)
                    if g.x.shape[1] > len(marker_names) and torch.any(g.x[:, len(marker_names):] != 0):
                        raise ValueError('Reference saved inputs contain missing fates; observed inputs are required.')
                    g.x = torch.as_tensor(exclusive_markers(g.x[:, :len(marker_names)].cpu().numpy(), marker_names), dtype=g.x.dtype)
                    physical = np.asarray(source['transform'].inverse(g.y.cpu().numpy())).reshape(-1) + np.asarray(source['baseline_offsets'][g.organoid_str]).reshape(-1)
                    baseline = np.asarray(source['baseline_predictions'][g.organoid_str]).reshape(-1)
                    offsets[g.organoid_str] = baseline.copy() if SETTINGS['residualize'] else np.zeros_like(baseline)
                    g.y = torch.as_tensor(physical - offsets[g.organoid_str], dtype=torch.float64)
                    g.full_num_cells = float(len(g.x))
                    if 'global_feat' in g:
                        del g.global_feat
                    groups[role].append(g)
                    if role == 'val':
                        baseline_validation.append(dict(organoid_str=g.organoid_str, n_cells=len(g.x), mse=float(np.mean((physical-baseline)**2))))
            prepared = dict(groups=groups, transform=IdentityTransform().fit(groups['train']),
                baseline=source['baseline'], baseline_predictions=source['baseline_predictions'], baseline_offsets=offsets,
                baseline_validation_mse=baseline_validation, baseline_features=source['baseline_features'],
                residualized=SETTINGS['residualize'], global_features=[], all_global_features=source['all_global_features'],
                global_center=source['global_center'], global_scale=source['global_scale'],
                source_run=str(reference.directory), source_input_bundle=reference_inputs[fold])
        baseline_rows.extend(dict(fold=fold, **row) for row in prepared['baseline_validation_mse'])
        pd.DataFrame(baseline_rows).to_csv(RUN_DIR / 'baseline_validation_mse.csv', index=False)
        input_bundle = f'inputs/fold_{fold}_all_intact'
        save_bundle(RUN_DIR / input_bundle, dict(**prepared, marker_names=marker_names), splits=split, provenance=provenance)
        samples = {}
        for role, group in prepared['groups'].items():
            samples[role] = []
            for g in group:
                oid = str(g.organoid_str)
                if oid not in count_cache:
                    entry = graph_samples([g], max(SETTINGS['radii']))[0]
                    count_cache[oid] = {k:v for k,v in entry.items() if k != 'y'}
                samples[role].append(dict(**count_cache[oid], y=g.y.cpu().numpy().reshape(-1).astype(float)))
        for variant, radius in model_grid:
            for seed in SETTINGS['seeds']:
                with progress.task(model=f'spline_{variant}', depth=radius, fold=fold, seed=seed):
                    model, metrics, trials = fit_spline(samples['train'], n_markers=len(marker_names), radius=radius,
                        variant=variant, n_splines=SETTINGS['n_splines'], smoothness_grid=SETTINGS['smoothness_grid'],
                        ridge=SETTINGS['ridge'], pair_ridge_grid=SETTINGS['pair_ridge_grid'],
                        inner_fraction=SETTINGS['inner_fraction'], seed=seed, blas_threads=SETTINGS['blas_threads'], callback=progress.stage)
                    key = f'spline_{variant}_d{radius}_f{fold}_s{seed}'
                    record = dict(key=key, name=f'spline_{variant}', variant=variant, depth=radius,
                        fold=fold, seed=seed, subset='all', signal='intact', bundle=f'models/{key}', input_bundle=input_bundle)
                    mse = sample_mse(model, samples['val'])
                    base = {row['organoid_str']:row['mse'] for row in prepared['baseline_validation_mse']}
                    for sample, error in zip(samples['val'], mse):
                        scores.append(dict(**record, organoid_str=sample['organoid_str'], n_cells=sample['N'], mse=float(error),
                            baseline_mse=base[sample['organoid_str']], mse_minus_baseline=float(error)-base[sample['organoid_str']]))
                    save_bundle(RUN_DIR / record['bundle'], dict(model=model, record=record, metrics=metrics,
                        history=trials.to_dict('list')), splits=split, provenance=provenance)
                    (RUN_DIR / record['bundle'] / 'fit_settings.json').write_text(json.dumps(metrics, indent=2))
                    records.append(record)
                    tuning_parts.append(trials.assign(key=key, fold=fold, seed=seed, variant=variant, depth=radius))
                    pd.DataFrame(records).to_json(RUN_DIR / 'models.json', orient='records', indent=2)
                    pd.DataFrame(scores).to_csv(RUN_DIR / 'validation_mse.csv', index=False)
                    pd.concat(tuning_parts, ignore_index=True).to_csv(RUN_DIR / 'penalty_trials.csv', index=False)
(RUN_DIR / 'complete.json').write_text(json.dumps({'models':len(records)}))
print('Saved run:', RUN_DIR)

## Validation performance and independent reload

MSE is in physical curvature squared, averaged equally across validation organoids. Repeated seeds/validation appearances are averaged within each organoid before calculating mean ± one SEM. These bands describe organoid variation conditional on fitted models, not coefficient uncertainty. The center-only result is displayed as a horizontal reference because it has no neighborhood radius. Baseline subtraction does not change MSE after reconstruction.

In [ ]:
scores_df = pd.read_csv(RUN_DIR / 'validation_mse.csv')
fig, ax = plt.subplots(figsize=(8, 4.5))
for label, frame in scores_df.groupby('name'):
    curve = organoid_mse_summary(frame, 'depth')
    if label == 'spline_center':
        x = [0, max(SETTINGS['radii'])]
        ax.plot(x, [curve['mean'].iloc[0]]*2, '--', label='Center only (radius 0)')
        ax.fill_between(x, [curve['mean'].iloc[0]-curve['sem'].iloc[0]]*2,
                        [curve['mean'].iloc[0]+curve['sem'].iloc[0]]*2, alpha=.12)
    else:
        line, = ax.plot(curve.depth, curve['mean'], 'o-', label=label)
        ax.fill_between(curve.depth, curve['mean']-curve['sem'], curve['mean']+curve['sem'], color=line.get_color(), alpha=.18)
b = organoid_mse_summary(pd.read_csv(RUN_DIR / 'baseline_validation_mse.csv')).iloc[0]
ax.axhline(b['mean'], color='black', linestyle=':', label='Global baseline')
ax.axhspan(b['mean']-b['sem'], b['mean']+b['sem'], color='black', alpha=.10)
ax.set(xlabel='Neighborhood radius (all shells up to r)', ylabel='Validation MSE (curvature²)', title='Mean ± SEM across organoids')
ax.set_xticks([0, *SETTINGS['radii']])
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'mse_by_radius.png', dpi=160)
plt.show()
saved = AnalysisRun(RUN_DIR)
restored = saved.select(saved.records.iloc[0].key)
print('Independently restored model and validation organoids:', restored['record']['key'], len(restored['groups']['val']))

## Regional validation MSE

Derive regions from the original graph crypt-distance arrays and circumference profiles, never another experiment's exported table. Missing or unqualified crypts remain separate. Crypt/neck/villus definitions and small-crypt detection limitations are the same as in GIN training. Predictions reconstruct physical curvature using each saved fold's baseline offsets. This section saves node annotations, regional support and MSE for use by the comparison notebook; it does not affect training.

In [ ]:
if EVALUATE_REGIONS:
    regional_run = AnalysisRun(RUN_DIR)
    region_output = RUN_DIR / 'regional_evaluation'
    region_output.mkdir(exist_ok=True)
    (region_output / 'settings.json').write_text(json.dumps(REGION_SETTINGS, indent=2))
    annotations, parts = {}, []
    for record in regional_run.records.to_dict('records'):
        selected = regional_run.select(record['key'])
        for g in selected['groups']['val']:
            if g.organoid_str not in annotations:
                annotations[g.organoid_str] = graph_regions(selected['raw_graphs'][g.organoid_str], ROOT/'training_data'/SETTINGS['dataset'], **REGION_SETTINGS)
        annotation = pd.concat([annotations[g.organoid_str] for g in selected['groups']['val']], ignore_index=True)
        table = regional_mse(prediction_table(selected, device='cpu'), annotation)
        for column in ('key', 'name', 'depth', 'fold', 'seed', 'subset', 'signal'):
            table[column] = record[column]
        parts.append(table)
    regional = pd.concat(parts, ignore_index=True)
    regional.to_csv(region_output / 'validation_mse.csv', index=False)
    all_annotations = pd.concat(annotations.values(), ignore_index=True)
    all_annotations.to_csv(region_output / 'node_regions.csv.gz', index=False)
    all_annotations.groupby(['region', 'annotation_status']).agg(cells=('node','size'), organoids=('organoid_str','nunique')).to_csv(region_output / 'support.csv')
    fig = plot_regional_mse(regional)
    fig.savefig(region_output / 'mse_by_radius.png', dpi=160, bbox_inches='tight')
    plt.show()